# Instrucciones para generar .json para evaluación

Estructura general: 
1. Guardar resultados de un modelo en un DataFrame de Pandas. 
2. Exportar ese DataFrame a un archivo .json.
3. Pasar ese archivo .json junto al archivo de evaluación a la paquetería de PyEvALL.

## Paso 1 y 2: Creación del DataFrame (RESULTADOS DE LOS MODELOS) y del archivo .json

En la vida real este DataFrame se obtiene de los resultados **REALES** del sistema que estemos trabajando. En este notebook creamos un dataframe sintético de manera aleatoria para que sirva de ejemplo de la estructura.

El dataframe debe de tener 3 columnas:
1. `test_case` Identificador para este task. SIEMPRE DEBE DE SER EL VALOR "EXIST2025"
2. `id` Valor numérico extraído del conjunto de datos inicial. Es importante que sea consistente ya que debe de emparejarse con el conjunto de datos que tiene los valores de verdad.
3. `value` La predicción del modelo. Hay dos variantes del posible valor aquí, en función del task. Favor de ver los detalles en el código.

El dataframe se debe de guardar usando la función `.to_json()` de pandas. IMPORTANTE: USEN EL PARÁMETRO `orient = 'records'`.

In [ ]:
# Creación de conjuntos de datos sintéticos.
import pandas as pd
import random

# Columna test_case
test_case = ["EXIST2025" for i in range(10)]

# Columna id
id = [f"30000{i}" for i in range(10)]

# Columna value: SUBTASK 2.1 Y 2.2
value = ["NO" if random.random() > 0.5 else "YES" for i in range(10)]

# Columna value: SUBTASK 2.3
rand_labels = ["NO", "IDEOLOGICAL-INEQUALITY", "STEREOTYPING-DOMINANCE", "OBJECTIFICATION", "SEXUAL-VIOLENCE", "MYSOGYNY-NON-SEXUAL-VIOLENCE"]
value_sub23 = [random.sample(rand_labels) for i in range(10)]


def create_and_json(test_list, id_list, value_list, sub_23):
    test_dictionary = {"test_case": test_list, "id": id_list, "value": value_list}
    df = pd.DataFrame(dataframe = test_dictionary)

    # Guardamos dos veces el mismo conjunto de datos ya que uno va a servir como el Gold Standard (dígase el conjunto de datos de prueba.)
    # USEN PARÁMETRO DE orient = 'records'
    # Pandas permite guardar los json bajo distintos formatos de orientación, pero esos formatos NO FUNCIONAN PARA PYEVALL.
    if not sub_23:
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.1\ y\ 2.2/pred/pred.json', orient = 'records')
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.1\ y\ 2.2/gold/gold.json', orient = 'records')
    else:
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.3/pred/pred.json', orient = 'records')
        df.to_json(f'/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/Subtask\ 2.3/gold/gold.json', orient = 'records')




## Paso 3: Uso de PyEvALL

En el pdf del EXIST 2026 viene una implementación de la evaluación (La misma que está aquí salvo unos ajustes)

In [5]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

def predicciones_papulince(pred_dir, gold_dir):
    orient = pred_dir.split('/')[-1]
    print("Trabajando con: {}".format(orient.split('.')[0][5:]))
    predictions = pred_dir
    gold = gold_dir
    test = PyEvALLEvaluation()
    params= dict()
    params[PyEvALLUtils.PARAM_REPORT]= PyEvALLUtils.PARAM_OPTION_REPORT_EMBEDDED
    metrics=["ICM", "ICMNorm" ,"FMeasure"]
    report= test.evaluate(predictions, gold, metrics, **params)
    report.print_report()
    print("===================================")

In [ ]:
predicciones_papulince('/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/pred/pred_orient_records.json', '/home/flopezp/GIL-Exist-2026/Archivo de salida/test_outputs/gold/orient_records.json')

Trabajando con: orient_columns
2026-04-22 13:34:42,693 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
{
  "metrics": {},
  "files": {
    "pred_orient_columns.json": {
      "name": "pred_orient_columns.json",
      "status": "FAIL",
      "gold": false,
      "description": "The file contains errors or warnings, please review them.\\nFile name: pred_orient_columns.json.",
      "errors": {
        "FORMAT_INCORRECT_SCHEMA_JSON_ERROR": {
          "description": "The file contains an incorrect json format according to the schema.\\nFile name: pred_orient_columns.json.\\nThe evaluation STOP.",
          "exception": "{'test_case': {'0': 'EXIST2025', '1': 'EXIST2025', '2': 'EXIST2025', '3': 'EXIST2025', '4': 'EXIST2025', '5': 'EXIST2025', '6': 'EXIST2025', '7': 'EXIST2025', '8': 'EXIST2025', '9': 'EXIST2025'}, 'id': {'0': '300000', '1': '300001', '2': '300002', '3': '300003', '4': '300004', '5': '300005', '6': '30000

In [ ]:
# Subtask 2.1/3.1
# Binary Classification - Mono Label
# Accuracy, System Precision, Kappa, Precision, Recall, F-Measure, ICM, ICM Norm
# OFFICIAL METRIC: ICM AND ICOM SOFT

# MetricFactory.Accuracy
# MetricFactory.SystemPrecision
# MetricFactory.Kappa
# MetricFactory.Precision
# MetricFactory.Recall
# MetricFactory.FMeasure
# MetricFactory.ICM
# MetricFactory.ICMNorm


# Subtask 2.2/3.2
# Multiclass Hierarchical Classification - Mono Label

# Subtask 2.3/3.3
# Multiclass Hierarchical Classification - Multi Label